In [ ]:
import requests

def get_nws_weather(lat: float, lon: float) -> dict:
    """Retrieves the weather forecast from the National Weather Service API for given coordinates.

    Args:
        lat (float): Latitude coordinate (e.g., 35.4676 for Oklahoma City)
        lon (float): Longitude coordinate (e.g., -97.5164 for Oklahoma City)

    Returns:
        dict: A dictionary containing the weather report or error status.
    """
    headers = {
        'User-Agent': '(myweatherbot.com, contact@myweatherbot.com)'
    }

    try:
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        response = requests.get(points_url, headers=headers)

        if response.status_code != 200:
            return {"status": "error", "error_message": f"Failed to fetch grid data: {response.status_code}"}

        data = response.json()
        forecast_url = data['properties']['forecast']

        forecast_response = requests.get(forecast_url, headers=headers)
        if forecast_response.status_code != 200:
            return {"status": "error", "error_message": "Failed to fetch forecast data."}

        forecast_data = forecast_response.json()
        period = forecast_data['properties']['periods'][0]

        report = f"{period['name']}: {period['detailedForecast']}"
        return {"status": "success", "report": report}

    except Exception as e:
        return {"status": "error", "error_message": str(e)}

# Test the function right here in the notebook
result = get_nws_weather(35.4676, -97.5164)
print(result)

{'status': 'success', 'report': 'Today: Mostly sunny, with a high near 90. South wind around 15 mph, with gusts as high as 23 mph.'}


In [ ]:
def weather_chat():
    print("Weather Bot: Hello! I am ready. Type 'okc' for Oklahoma City, or enter coordinates like '38.8951, -77.0364'. Type 'exit' to quit.")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() == 'exit':
            print("Weather Bot: Goodbye!")
            break

        if user_input.lower() == 'okc':
            lat, lon = 35.4676, -97.5164
            print(f"Weather Bot: Fetching weather for Oklahoma City...")
        else:
            try:
                parts = user_input.split(',')
                lat = float(parts[0].strip())
                lon = float(parts[1].strip())
            except Exception:
                print("Weather Bot: Please type 'okc' or enter valid latitude, longitude (e.g., 35.4676, -97.5164)")
                continue

        # Call your existing function
        result = get_nws_weather(lat, lon)
        if result["status"] == "success":
            print(f"Weather Bot: {result['report']}\n")
        else:
            print(f"Weather Bot: Error - {result['error_message']}\n")

# Run the interactive chat
weather_chat()

Weather Bot: Hello! I am ready. Type 'okc' for Oklahoma City, or enter coordinates like '38.8951, -77.0364'. Type 'exit' to quit.
You: okc
Weather Bot: Fetching weather for Oklahoma City...
Weather Bot: Today: Mostly sunny, with a high near 90. South wind around 15 mph, with gusts as high as 23 mph.

You: exit
Weather Bot: Goodbye!


In [ ]:
import requests

def get_lat_lon_from_place(place_name: str) -> dict:
    """Converts a place name or address into latitude and longitude coordinates using a public geocoding service.

    Args:
        place_name (str): The city, address, or location name (e.g., "Chicago, IL")

    Returns:
        dict: A dictionary containing the latitude and longitude, or an error status.
    """
    # Using a public open-source geocoding API endpoint for seamless lab testing without requiring a paid key
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={requests.utils.quote(place_name)}&count=1&format=json"

    try:
        response = requests.get(url)
        data = response.json()

        if "results" in data and len(data["results"]) > 0:
            loc = data["results"][0]
            return {
                "status": "success",
                "lat": loc["latitude"],
                "lon": loc["longitude"]
            }
        else:
            return {"status": "error", "error_message": f"Could not find coordinates for {place_name}"}

    except Exception as e:
        return {"status": "error", "error_message": str(e)}

In [ ]:
# Install the Google Agent Development Kit
!pip install google-adk -q

from google.adk import Agent

# Combine both functions into a tools list for the agent
weather_tools = [get_lat_lon_from_place, get_nws_weather]

# Initialize the ADK Agent
weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    tools=weather_tools,
    instruction="""
    You are an expert meteorological weather assistant.
    1. When a user asks for the weather in a specific city, use the 'get_lat_lon_from_place' tool first to find its coordinates.
    2. Once you have the coordinates, use the 'get_nws_weather' tool to fetch the NWS forecast.
    3. Present the forecast clearly and concisely to the user.
    """
)

print("ADK Weather Agent successfully initialized with tools!")

ADK Weather Agent successfully initialized with tools!


In [ ]:
# Install LiteLLM for multi-model provider support
!pip install litellm -q

# Initialize an alternative agent using a third-party model configuration route
third_party_weather_agent = Agent(
    name="claude_weather_agent",
    model="claude-3-5-sonnet", # Handled via ADK/LiteLLM provider routing
    tools=weather_tools,
    instruction="You are a helpful weather assistant powered by alternative model routing using the same geocoding and NWS tools.",
)

print("Third-party model agent configured successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.8 MB/s eta 0:00:00
Third-party model agent configured successfully!


In [ ]:
def test_multiple_cities():
    cities = ["Seattle, WA", "Denver, CO", "Miami, FL", "Boston, MA"]

    print("--- Running Multi-City Test Suite ---")
    for city in cities:
        print(f"\nProcessing target city: {city}")

        # 1. Test Geocoding Tool
        geo = get_lat_lon_from_place(city)
        if geo["status"] == "success":
            lat, lon = geo["lat"], geo["lon"]
            print(f"  -> Resolved Lat/Lon: {lat}, {lon}")

            # 2. Test NWS Weather Tool
            weather = get_nws_weather(lat, lon)
            if weather["status"] == "success":
                print(f"  -> Forecast: {weather['report']}")
            else:
                print(f"  -> NWS Error: {weather['error_message']}")
        else:
            print(f"  -> Geocoding Error: {geo['error_message']}")

# Run the test suite
test_multiple_cities()

--- Running Multi-City Test Suite ---

Processing target city: Seattle, WA
  -> Resolved Lat/Lon: 47.60621, -122.33207
  -> Forecast: Today: Mostly cloudy, with a high near 60. South southeast wind around 3 mph.

Processing target city: Denver, CO
  -> Resolved Lat/Lon: 39.73915, -104.9847
  -> Forecast: Today: Areas of drizzle and patchy fog before noon. Mostly cloudy. High near 65, with temperatures falling to around 62 in the afternoon. North northeast wind around 8 mph. Chance of precipitation is 20%.

Processing target city: Miami, FL
  -> Resolved Lat/Lon: 25.77427, -80.19366
  -> Forecast: Today: Sunny, with a high near 87. Northeast wind around 15 mph, with gusts as high as 18 mph.

Processing target city: Boston, MA
  -> Resolved Lat/Lon: 42.35843, -71.05977
  -> Forecast: Today: Patchy fog and a slight chance of rain showers. Cloudy, with a high near 59. Southeast wind around 8 mph. Chance of precipitation is 20%. New rainfall amounts less than a tenth of an inch possible.
